# 应用负载数据分析

## 目标
- 探索性数据分析（EDA）
- 识别负载模式和规律
- 为模型训练准备数据

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# 添加项目路径
sys.path.append(os.path.abspath('..'))

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 忽略警告
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 从本地模块导入
from src.preprocessing.data_loader import DataLoader
from src.preprocessing.preprocessor import DataPreprocessor
from src.visualization.plotter import LoadVisualizer

print("导入成功！")

## 1. 加载数据

In [ ]:
# 初始化数据加载器
loader = DataLoader(data_dir='../data/raw')

# TODO: 修改为您的数据文件名
# df = loader.load_csv('your_data.csv')

# 示例：创建模拟数据（实际使用时请删除此部分）
dates = pd.date_range('2024-01-01', periods=365*24, freq='H')
np.random.seed(42)
load_values = 100 + 20 * np.sin(2 * np.pi * dates.hour / 24) + \
              10 * np.sin(2 * np.pi * dates.dayofweek / 7) + \
              np.random.normal(0, 5, len(dates))

df = pd.DataFrame({
    'timestamp': dates,
    'cpu_usage': load_values,
    'memory_usage': load_values * 1.2,
    'request_count': (load_values * 10).astype(int)
})

print(f"数据形状：{df.shape}")
df.head()

## 2. 数据概览

In [ ]:
# 基本信息
print("数据集信息:")
print(df.info())
print("\n描述性统计:")
print(df.describe())
print("\n缺失值统计:")
print(df.isnull().sum())

## 3. 数据可视化探索

In [ ]:
# 初始可视化工具
visualizer = LoadVisualizer()

# 绘制 CPU 使用率趋势
visualizer.plot_load_trend(
    df.head(168),  # 显示一周的数据
    'timestamp', 
    'cpu_usage', 
    'CPU 使用率趋势图（最近 7 天）'
)

In [ ]:
# 绘制热力图
visualizer.plot_heatmap_by_hour(df, 'timestamp', 'cpu_usage')

In [ ]:
# 绘制分布图
visualizer.plot_distribution(df, 'cpu_usage', 'CPU 使用率分布')

## 4. 数据预处理

In [ ]:
# 初始化预处理器
preprocessor = DataPreprocessor()

# 数据清洗
df_clean = preprocessor.clean_data(df)
print(f"清洗后数据形状：{df_clean.shape}")

In [ ]:
# 提取时间特征
df_processed = preprocessor.extract_time_features(df_clean, 'timestamp')
print("时间特征提取完成")
print(df_processed[['timestamp', 'hour', 'day_of_week', 'is_weekend', 'is_peak_hour']].head(10))

In [ ]:
# 创建滞后特征和滚动特征
df_processed = preprocessor.create_lag_features(df_processed, 'cpu_usage', lag_periods=[1, 2, 3, 24])
df_processed = preprocessor.create_rolling_features(df_processed, 'cpu_usage', windows=[3, 6, 12, 24])

print("特征工程完成")
print(f"最终特征数量：{df_processed.shape[1]}")

## 5. 相关性分析

In [ ]:
# 绘制相关性矩阵
visualizer.plot_correlation_matrix(df_processed, '特征相关性分析')

## 6. 保存处理后的数据

In [ ]:
# 保存处理后的数据
loader.save_processed(df_processed, 'processed_load_data.csv')
print("\n数据已保存，可以进行模型训练了！")